In [1]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

# Asegurar que la raíz del proyecto esté en sys.path
repo_root = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.common.config_loader import load_config
from src.common.spark_session import get_spark_session

config = load_config()
spark = get_spark_session(app_name="ExploreLocalDelta", is_local=True)
print("Spark local listo.")

Spark local listo.


In [2]:
df_silver = spark.read.format("delta").load(f"{repo_root}/data/silver/flights_silver")
df_silver_metrics = spark.read.format("delta").load(f"{repo_root}/data/silver/silver_quality_metrics")

print(f"Total filas en flights_silver: {df_silver.count():,}")
df_silver.printSchema()
df_silver.show(5, truncate=False)
df_silver_metrics.show(truncate=False)

Total filas en flights_silver: 5,819,079
root
 |-- dest_airport_code: string (nullable = true)
 |-- origin_airport_code: string (nullable = true)
 |-- airline_code: string (nullable = true)
 |-- raw_dest_airport: string (nullable = true)
 |-- raw_origin_airport: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- flight_number: integer (nullable = true)
 |-- tail_number: string (nullable = true)
 |-- scheduled_departure_hhmm: integer (nullable = true)
 |-- scheduled_arrival_hhmm: integer (nullable = true)
 |-- scheduled_time_min: double (nullable = true)
 |-- distance_miles: double (nullable = true)
 |-- departure_time_hhmm: integer (nullable = true)
 |-- departure_delay_min: double (nullable = true)
 |-- taxi_out_min: double (nullable = true)
 |-- wheels_off_hhmm: integer (nullable = true)
 |-- elapsed_time_min: d

In [3]:
gold_star_tables = ["dim_date", "dim_airline", "dim_airport", "dim_weather_condition", "fact_flights"]

for tbl in gold_star_tables:
    df = spark.read.format("delta").load(f"{repo_root}/data/gold/{tbl}")
    print(f"\n=== GOLD A: {tbl} ({df.count():,} filas) ===")
    df.show(5, truncate=False)


=== GOLD A: dim_date (365 filas) ===
+--------+-----------+----+-----+---+-----------+--------+----------+-------+----------+-------------+--------------+
|date_sk |flight_date|year|month|day|day_of_week|day_name|month_name|quarter|is_weekend|is_us_holiday|holiday_name  |
+--------+-----------+----+-----+---+-----------+--------+----------+-------+----------+-------------+--------------+
|20150101|2015-01-01 |2015|1    |1  |4          |Thursday|January   |1      |false     |true         |New Year's Day|
|20150102|2015-01-02 |2015|1    |2  |5          |Friday  |January   |1      |false     |false        |No Holiday    |
|20150103|2015-01-03 |2015|1    |3  |6          |Saturday|January   |1      |true      |false        |No Holiday    |
|20150104|2015-01-04 |2015|1    |4  |7          |Sunday  |January   |1      |true      |false        |No Holiday    |
|20150105|2015-01-05 |2015|1    |5  |1          |Monday  |January   |1      |false     |false        |No Holiday    |
+--------+--------

In [4]:
df_ml = spark.read.format("delta").load(f"{repo_root}/data/gold/ml_flight_delay_training")
df_ml_audit = spark.read.format("delta").load(f"{repo_root}/data/gold/gold_b_audit_log")

print(f"Total filas en ml_flight_delay_training: {df_ml.count():,}")
df_ml.printSchema()
df_ml.show(5, truncate=False)

print("=== Auditoría Gold B ===")
df_ml_audit.show(truncate=False)

print("=== Balance de clases por partición temporal ===")
(
    df_ml.groupBy("dataset_split")
    .agg(
        F.count("*").alias("total_rows"),
        F.round(F.avg("target_is_delayed_15") * 100, 2).alias("pct_delayed_15"),
    )
    .orderBy("dataset_split")
    .show()
)

Total filas en ml_flight_delay_training: 4,652,278
root
 |-- flight_sk: long (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- dataset_split: string (nullable = true)
 |-- target_is_delayed_15: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_us_holiday: integer (nullable = true)
 |-- airline_code: string (nullable = true)
 |-- origin_airport_code: string (nullable = true)
 |-- dest_airport_code: string (nullable = true)
 |-- scheduled_dep_hour: integer (nullable = true)
 |-- scheduled_time_min: double (nullable = true)
 |-- distance_miles: double (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- airline_hist_delay_rate: double (

## Auditoria de sanidad sobre Gold A

In [5]:
from pyspark.sql import functions as F

df_fact = spark.read.format("delta").load(f"{repo_root}/data/gold/fact_flights")
df_date = spark.read.format("delta").load(f"{repo_root}/data/gold/dim_date")
df_airline = spark.read.format("delta").load(f"{repo_root}/data/gold/dim_airline")
df_airport = spark.read.format("delta").load(f"{repo_root}/data/gold/dim_airport")
df_weather = spark.read.format("delta").load(f"{repo_root}/data/gold/dim_weather_condition")

# 1. Integridad referencial estricta (todos los conteos deben dar exactamente 0)
orphans = {
    "orphan_date_sk": df_fact.join(df_date, "date_sk", "left_anti").count(),
    "orphan_airline_sk": df_fact.join(df_airline, "airline_sk", "left_anti").count(),
    "orphan_origin_airport_sk": df_fact.join(df_airport, df_fact.origin_airport_sk == df_airport.airport_sk, "left_anti").count(),
    "orphan_dest_airport_sk": df_fact.join(df_airport, df_fact.dest_airport_sk == df_airport.airport_sk, "left_anti").count(),
    "orphan_weather_sk": df_fact.join(df_weather, "weather_sk", "left_anti").count(),
}
print("Claves huérfanas en fact_flights (deben ser 0):", orphans)

# 2. Coherencia de estado operativo (cancelados/desviados vs completados)
(
    df_fact.groupBy("is_cancelled", "is_diverted")
    .agg(
        F.count("*").alias("total_flights"),
        F.count("arrival_delay_min").alias("non_null_arr_delay"),
        F.round(F.avg("arrival_delay_min"), 2).alias("avg_arr_delay"),
        F.round(F.avg("is_delayed_15") * 100, 2).alias("pct_delayed_15"),
    )
    .show()
)

# 3. Distribución de categorías climáticas (incluyendo weather_sk = -1)
(
    df_fact.join(df_weather, "weather_sk")
    .groupBy("weather_sk", "precipitation_category", "wind_category", "is_weather_observed")
    .agg(
        F.count("*").alias("flights"),
        F.round(F.avg("departure_delay_min"), 2).alias("avg_dep_delay"),
    )
    .orderBy(F.desc("flights"))
    .show(truncate=False)
)

Claves huérfanas en fact_flights (deben ser 0): {'orphan_date_sk': 0, 'orphan_airline_sk': 0, 'orphan_origin_airport_sk': 0, 'orphan_dest_airport_sk': 0, 'orphan_weather_sk': 0}
+------------+-----------+-------------+------------------+-------------+--------------+
|is_cancelled|is_diverted|total_flights|non_null_arr_delay|avg_arr_delay|pct_delayed_15|
+------------+-----------+-------------+------------------+-------------+--------------+
|       false|       true|        15187|                 0|         NULL|          NULL|
|        true|      false|        89884|                 0|         NULL|          NULL|
|       false|      false|      5714008|           5714008|         4.41|         18.61|
+------------+-----------+-------------+------------------+-------------+--------------+

+----------+----------------------+----------------------------+-------------------+-------+-------------+
|weather_sk|precipitation_category|wind_category               |is_weather_observed|flights

Integridad referencial perfecta: 0 claves huérfanas en las 5 relaciones (date_sk, airline_sk, origin_airport_sk, dest_airport_sk, weather_sk).

Estados operativos limpios: Los 15.187 desviados y 89.884 cancelados tienen 0 valores no nulos en arrival_delay_min y is_delayed_15 (NULL), mientras que los 5.714.008 vuelos completados tienen el 100% de sus métricas completas con una tasa base de demora de 18.61%.

Curva física del clima: El retraso promedio de salida (avg_dep_delay) escala de forma totalmente monotónica y realista con la severidad climática:

    Seco + viento leve: 8.56 min

    Seco + viento moderado / fuerte: 10.98 min -> 13.63 min

    Lluvia leve + viento leve / moderado / fuerte: 15.09 min -> 17.02 min -> 24.45 min

    Lluvia moderada/fuerte + viento leve / moderado / fuerte: 29.15 min -> 33.26 min -> 55.38 min

    Nieve + viento leve / moderado / fuerte (ventisca): 27.42 min -> 40.19 min -> 86.47 min

## Auditoría sanidad y anti-leakage Gold B

In [6]:
df_ml = spark.read.format("delta").load(f"{repo_root}/data/gold/ml_flight_delay_training")

# 1. Conteo de nulos por columna (todas las features deben dar 0 nulos)
null_counts = df_ml.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_ml.columns])
null_counts.show(vertical=True)

# 2. Resumen estadístico de variables numéricas (verificar mínimos y máximos físicos)
numeric_cols = [
    "target_is_delayed_15",
    "scheduled_dep_hour",
    "scheduled_time_min",
    "distance_miles",
    "temperature_2m",
    "precipitation",
    "snowfall",
    "wind_speed_10m",
    "wind_gusts_10m",
    "cloud_cover",
    "airline_hist_delay_rate",
    "origin_hist_delay_rate",
]
df_ml.select(numeric_cols).summary("min", "mean", "50%", "max").show(vertical=True)

# 3. Verificación anti-leakage: verificar que el primer día del año (2015-01-01) use el prior (0.19)
# y que la correlación con el target sea razonable (ni 0 ni > 0.90)
df_ml.filter(F.col("flight_date") == "2015-01-01").select(
    "airline_code", "flight_date", "airline_hist_delay_rate", "origin_hist_delay_rate"
).distinct().show(5)

print("Correlación airline_hist_delay_rate vs target:", round(df_ml.stat.corr("airline_hist_delay_rate", "target_is_delayed_15"), 4))
print("Correlación precipitation vs target:", round(df_ml.stat.corr("precipitation", "target_is_delayed_15"), 4))
print("Correlación wind_speed_10m vs target:", round(df_ml.stat.corr("wind_speed_10m", "target_is_delayed_15"), 4))

-RECORD 0-------------------------
 flight_sk                  | 0   
 flight_date                | 0   
 dataset_split              | 0   
 target_is_delayed_15       | 0   
 month                      | 0   
 day_of_week                | 0   
 is_weekend                 | 0   
 is_us_holiday              | 0   
 airline_code               | 0   
 origin_airport_code        | 0   
 dest_airport_code          | 0   
 scheduled_dep_hour         | 0   
 scheduled_time_min         | 0   
 distance_miles             | 0   
 temperature_2m             | 0   
 precipitation              | 0   
 snowfall                   | 0   
 wind_speed_10m             | 0   
 wind_gusts_10m             | 0   
 cloud_cover                | 0   
 airline_hist_delay_rate    | 0   
 airline_hist_avg_dep_delay | 0   
 origin_hist_delay_rate     | 0   
 origin_hist_avg_dep_delay  | 0   

-RECORD 0---------------------------------------
 summary                 | min                  
 target_is_delayed_15    |

Completitud total: 0 nulos en las 24 columnas sobre 4.652.278 filas.

Rangos físicos realistas:

    Temperatura mínima -30.4 °C (invierno en Minnesota/Dakota) y máxima 45.5 °C (verano en Phoenix/Las Vegas).

    Distancia mínima 62 millas (saltos regionales cortos) y máxima 4.983 millas (rutas JFK–HNL / Newark–Honolulu).

    Ráfagas máximas 102.6 km/h y nubosidad 0–100%.

Anti-leakage confirmado: El 2015-01-01 arranca exactamente en el prior 0.19 para todas las aerolíneas y aeropuertos, y la tasa histórica acumulada tiene una correlación positiva limpia (+0.0959) con el target, al igual que precipitación (+0.0837) y viento (+0.0487).